# 01 Explore: IBM AML transactions

Work card 01. Loads `data/HI-Small_Trans.csv`, checks memory, splits by time and looks at fraud rate and fan-in.

In [1]:
import sys, time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_transactions, time_split, split_summary

!free -h
!nproc

/usr/bin/sh: line 1: free: command not found
3


Set `NROWS` to a number (e.g. `1_000_000`) if memory is tight. `None` loads everything.

In [2]:
NROWS = None

start = time.time()
df = load_transactions(ROOT / 'data' / 'HI-Small_Trans.csv', nrows=NROWS)
print(f'Loaded {len(df):,} rows in {time.time() - start:.1f}s')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
df.head()

Loaded 5,078,345 rows in 14.1s
Memory: 0.80 GB


,ts,amount,receiving_currency,amount_paid,payment_currency,payment_format,label,src,dst
0,2022-09-01,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,03209_8000F4670,03209_8000F4670
1,2022-09-01,897.37,US Dollar,897.37,US Dollar,Reinvestment,0,01420_8005DFEB0,01420_8005DFEB0
2,2022-09-01,99986.94,US Dollar,99986.94,US Dollar,Reinvestment,0,010_8000F6850,010_8000F6850
3,2022-09-01,16.08,US Dollar,16.08,US Dollar,Reinvestment,0,012_8001167D0,012_8001167D0
4,2022-09-01,10.30,US Dollar,10.30,US Dollar,Reinvestment,0,001_8001364A0,001_8001364A0


In [3]:
df.dtypes

ts                    datetime64[us]
amount                       float64
receiving_currency          category
amount_paid                  float64
payment_currency            category
payment_format              category
label                           int8
src                           string
dst                           string
dtype: object

## Time split

The test split must be the latest dates.

In [4]:
train, val, test = time_split(df)
split_summary(train, val, test)

,split,rows,start,end,fraud_rows,fraud_rate_%
0,train,5077237,2022-09-01 00:00:00,2022-09-10 23:59:00,4522,0.0891
1,val,861,2022-09-11 00:05:00,2022-09-13 23:59:00,508,59.0012
2,test,247,2022-09-14 00:29:00,2022-09-18 16:18:00,147,59.5142


## Volume and fraud per day

Check whether the last days are very sparse. If so, the split may need adjusting.

In [5]:
per_day = df.groupby(df['ts'].dt.date).agg(rows=('label', 'size'), fraud=('label', 'sum'))
per_day['fraud_rate_%'] = (100 * per_day['fraud'] / per_day['rows']).round(4)
per_day

,rows,fraud,fraud_rate_%
ts,,,
2022-09-01,1114921,322,0.0289
2022-09-02,754449,408,0.0541
2022-09-03,207382,391,0.1885
2022-09-04,207430,407,0.1962
2022-09-05,482650,471,0.0976
2022-09-06,482089,531,0.1101
2022-09-07,482751,497,0.1030
2022-09-08,482773,539,0.1116
2022-09-09,654467,514,0.0785


## Currencies and payment formats

Amounts are in mixed currencies; card 02 must decide how to handle this.

In [6]:
print(df['payment_currency'].value_counts().head(15))
print()
print(df.groupby('payment_format', observed=True)['label'].agg(['size', 'mean']).sort_values('mean', ascending=False))

payment_currency
US Dollar            1895172
Euro                 1168297
Swiss Franc           234860
Yuan                  213752
Shekel                192184
Rupee                 190202
UK Pound              180738
Yen                   155209
Ruble                 155178
Bitcoin               146066
Canadian Dollar       140042
Australian Dollar     136769
Mexican Peso          110159
Saudi Riyal            89014
Brazil Real            70703
Name: count, dtype: int64

                   size      mean
payment_format                   
ACH              600797  0.007462
Bitcoin          146091  0.000383
Cash             490891  0.000220
Cheque          1864331  0.000174
Credit Card     1323324  0.000156
Reinvestment     481056  0.000000
Wire             171855  0.000000


## Fan-in on the training period

Receivers that collect from many different senders. Do they receive more laundering transactions?

In [7]:
fan_in = train.groupby('dst').agg(
    unique_senders=('src', 'nunique'),
    received=('label', 'size'),
    laundering=('label', 'sum'),
)
fan_in['laundering_share'] = fan_in['laundering'] / fan_in['received']
fan_in.sort_values('unique_senders', ascending=False).head(20)

,unique_senders,received,laundering,laundering_share
dst,,,,
070_100428660,545,1084,0,0.000000
070_1004286A8,328,653,0,0.000000
021611_8051A3FA0,93,94,0,0.000000
004726_808FAF7C0,90,90,0,0.000000
0014277_80B847A90,85,85,0,0.000000
013145_80640B300,83,83,0,0.000000
0039678_80F523260,83,84,1,0.011905
02915_801AF5660,83,83,1,0.012048
007042_80640FE10,83,83,0,0.000000


In [8]:
bins = [0, 1, 2, 5, 10, 50, float('inf')]
fan_in['fan_in_band'] = pd.cut(fan_in['unique_senders'], bins=bins)
fan_in.groupby('fan_in_band', observed=True).agg(receivers=('received', 'size'), laundering_share=('laundering_share', 'mean'))

,receivers,laundering_share
fan_in_band,,
"(0.0, 1.0]",161887,0.000291
"(1.0, 2.0]",89168,0.002281
"(2.0, 5.0]",157053,0.001419
"(5.0, 10.0]",9665,0.004393
"(10.0, 50.0]",2585,0.011690
"(50.0, inf]",267,0.001124
